### — Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import pandas as pd
import plotly.express as px
from pathlib import Path
from dotenv import load_dotenv

sys.path.append(str(Path().resolve().parent))
from src.scraper import scrape_all_cities
from src.scraper import enrich_hotels_details

load_dotenv(dotenv_path=Path().resolve().parent / ".env")
print("Imports OK")

### — Chargement des villes (output notebook 01)

In [ ]:
df_cities = pd.read_csv(Path().resolve().parent / "data" / "raw" / "cities.csv")
print(f"{len(df_cities)} villes chargées")
df_cities[["city_id", "city", "weather_score", "rank"]].head()

### — Paramètres

In [ ]:
CHECKIN             = "2026-09-25"
CHECKOUT            = "2026-09-26"
#MAX_HOTELS_PER_CITY = 20

In [ ]:
# Test sur une ville. Exemple Lille

df_lille = scrape_all_cities(
    df_cities[df_cities["city"] == "Lille"],
    max_hotels_per_city=None,
    force_refresh=True,
    headless=False
)

###  — Scraping complet 

In [ ]:
# Cellule 5 — Scraping complet 35 villes
df_hotels = scrape_all_cities(
    df_cities,
    max_hotels_per_city=None,
    headless=True,
    delay_range=(2, 4),
    force_refresh=True
)

print(f"Total : {len(df_hotels)} hôtels")
print(f"\nValeurs manquantes :\n{df_hotels.isnull().sum()}")
df_hotels.head(3)

In [ ]:
# Sauvegarder le scraping brut

df_hotels_raw = df_hotels
out = Path().resolve().parent / "data" / "raw" / "hotels_search.csv"
df_hotels_raw.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Sauvegardé : {out}")

In [ ]:
# Drop missing values for reviews
df_hotels_raw = df_hotels_raw.dropna(
    subset=["review_count"]
)

#### 🏆 Calcul du score de ranking (Moyenne Bayésienne)

On utilise une formule bayésienne pour calculer un score de classement (*ranking*) pertinent. Cette méthode permet de **pénaliser les hôtels ayant une excellente note mais un trop faible nombre d'avis**.

La formule mathématique appliquée est la suivante :

$$\text{hotel\_rank} = \left( \frac{\text{review\_count}}{\text{review\_count} + M} \times \text{score} \right) + \left( \frac{M}{\text{review\_count} + M} \times C \right)$$

---

##### 📊 Signification des variables :
* **`score`** : Note brute attribuée à l'hôtel (ex: 8.7, 9.1).
* **`review_count`** : Nombre d'avis / expériences vécues pour cet hôtel.
* **`C`** : Note moyenne globale de tous les hôtels du jeu de données (`df_hotels["score"].mean()`).
* **`M` (`GLOBAL_M = 1000`)** : Seuil de confiance (le nombre d'avis minimal théorique requis pour qu'une note brute soit jugée totalement fiable).

In [ ]:
# Calculer un score Bayesien de ranking selon le nombrer de review et le score brut de booking.com

GLOBAL_M = 1000 # Nombre reviews moyen
C = df_hotels_raw["score"].mean()

# On utilise .loc[:, "nom_colonne"] pour une assignation sécurisée
df_hotels_raw.loc[:, "hotel_rank"] = (
    (df_hotels_raw["review_count"] / (df_hotels_raw["review_count"] + GLOBAL_M)) * df_hotels_raw["score"]
    +
    (GLOBAL_M / (df_hotels_raw["review_count"] + GLOBAL_M)) * C
)

In [ ]:
# Prendre uniquement top 20 hotels par ville selon le noveau ranking
df_hotels_raw_top20 = (
    df_hotels_raw
    .sort_values(
        ["city", "hotel_rank"],
        ascending=[True, False]
    )
    .groupby("city", group_keys=False)
    .head(20)
    .reset_index(drop=True)
)

# Sauvegarder le top 20 hotels par ville

out = Path().resolve().parent / "data" / "processed" / "hotels_top20_ranking.csv"
df_hotels_raw_top20.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Sauvegardé : {out}")

In [ ]:
# Enrichir hotels de lat, lon, description

df_hotels_final = enrich_hotels_details(
    df_hotels_raw_top20,
    headless=False
)

In [ ]:
# Fill distance nan values with meaningful values

df_hotels_final["distance"] = (
    df_hotels_final["distance"]
    .fillna("Non renseignée")
)

### — Sauvegarde 

### — Nettoyage rapide avant sauvegarde finale

In [ ]:
# suppression lignes dont nom manquant

df_hotels_final = df_hotels_final.dropna(subset=["hotel_name"]).reset_index(drop=True)
print(f"Après nettoyage : {df_hotels_final.shape}")
df_hotels_final.isnull().sum()

### Sauvegarde finale de la version nettoyée :

In [ ]:
out = Path().resolve().parent / "data" / "raw" / "hotels.csv"
df_hotels_final.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Sauvegardé : {out}")